In [1]:
# ============================================================
# ETAPA 2 — BLOCO 1: Setup do PyTorch e verificação da GPU
# ============================================================
!pip install medmnist -q

import numpy as np
import torch
import torch.nn as nn

# Verificar se a GPU foi reconhecida
print("Versão do PyTorch:", torch.__version__)
print("GPU disponível?", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Nome da GPU:", torch.cuda.get_device_name(0))
    print("Número de GPUs:", torch.cuda.device_count())

# Definir o 'device' que usaremos (GPU se houver, senão CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("\nUsando device:", device)

# Fixar seeds para reprodutibilidade (mesma lógica da Etapa 1)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 93.6 MB/s eta 0:00:00:00:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.


In [2]:
# ============================================================
# ETAPA 2 — BLOCO 2: Carregar e preparar dados (28×28 achatado)
# ============================================================
from medmnist import PathMNIST

# Baixar os splits (mesma versão da Etapa 1: 28×28)
train_set = PathMNIST(split="train", size=28, download=True)
val_set   = PathMNIST(split="val",   size=28, download=True)
test_set  = PathMNIST(split="test",  size=28, download=True)

# Mesma preparação da Etapa 1: achatar + normalizar
def preparar_dados(dataset):
    x = dataset.imgs                      # (N, 28, 28, 3)
    y = dataset.labels                    # (N, 1)
    x = x.reshape(x.shape[0], -1)         # (N, 2352)
    x = x.astype(np.float32) / 255.0      # normaliza [0,1]
    y = y.flatten()                       # (N,)
    return x, y

x_train, y_train = preparar_dados(train_set)
x_val,   y_val   = preparar_dados(val_set)
x_test,  y_test  = preparar_dados(test_set)

# Converter de NumPy para tensores do PyTorch
# (no PyTorch tudo é tensor; é o equivalente do array do NumPy, mas roda na GPU)
X_train_t = torch.from_numpy(x_train)
y_train_t = torch.from_numpy(y_train).long()  # rótulos como inteiros (long)
X_val_t   = torch.from_numpy(x_val)
y_val_t   = torch.from_numpy(y_val).long()

print("Shapes dos tensores:")
print("X_train_t:", X_train_t.shape, "| dtype:", X_train_t.dtype)
print("y_train_t:", y_train_t.shape, "| dtype:", y_train_t.dtype)
print("X_val_t:  ", X_val_t.shape)
print("\nClasses:", torch.unique(y_train_t))

100%|██████████| 206M/206M [00:10<00:00, 19.5MB/s] 


Shapes dos tensores:
X_train_t: torch.Size([89996, 2352]) | dtype: torch.float32
y_train_t: torch.Size([89996]) | dtype: torch.int64
X_val_t:   torch.Size([10004, 2352])

Classes: tensor([0, 1, 2, 3, 4, 5, 6, 7, 8])


In [3]:
# ============================================================
# ETAPA 2 — BLOCO 3: A MLP em PyTorch
# ============================================================

class MLP_PyTorch(nn.Module):
    def __init__(self, tamanhos):
        """
        tamanhos: lista de neurônios por camada, ex.: [2352, 256, 128, 9]
        Mesma arquitetura da Etapa 1.
        """
        super().__init__()
        camadas = []
        for i in range(len(tamanhos) - 1):
            # Camada linear: equivale ao "z = a @ W + b" da Etapa 1
            camadas.append(nn.Linear(tamanhos[i], tamanhos[i + 1]))
            # ReLU em todas menos a última camada
            if i < len(tamanhos) - 2:
                camadas.append(nn.ReLU())
        # nn.Sequential encadeia as camadas na ordem
        self.rede = nn.Sequential(*camadas)

    def forward(self, x):
        # Só o forward! A backward é automática (autograd).
        # NÃO aplicamos softmax aqui de propósito (ver nota abaixo).
        return self.rede(x)


# ----- Criar o modelo e enviar para a GPU -----
modelo = MLP_PyTorch([2352, 256, 128, 9]).to(device)
print(modelo)

# Contar parâmetros (deve bater com a Etapa 1: ~310k para [2352,256,128,9])
total_params = sum(p.numel() for p in modelo.parameters())
print(f"\nTotal de parâmetros treináveis: {total_params:,}")

MLP_PyTorch(
  (rede): Sequential(
    (0): Linear(in_features=2352, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=128, bias=True)
    (3): ReLU()
    (4): Linear(in_features=128, out_features=9, bias=True)
  )
)

Total de parâmetros treináveis: 636,425


In [4]:
# ============================================================
# ETAPA 2 — BLOCO 4: Loss, Otimizador e Loop de Treino
# ============================================================
import torch.optim as optim

# ---- Hiperparâmetros (MESMOS da Etapa 1, para equivalência) ----
EPOCAS     = 20
BATCH_SIZE = 128
LR         = 0.01
BETA       = 0.9   # momentum

# ---- Função de perda ----
# CrossEntropyLoss já inclui o Softmax internamente (estável).
criterio = nn.CrossEntropyLoss()

# ---- Otimizador: SGD com Momentum (mesmo da Etapa 1) ----
otimizador = optim.SGD(modelo.parameters(), lr=LR, momentum=BETA)

# ---- Enviar os dados de treino para a GPU ----
X_train_dev = X_train_t.to(device)
y_train_dev = y_train_t.to(device)
X_val_dev   = X_val_t.to(device)
y_val_dev   = y_val_t.to(device)

# ---- Função de acurácia ----
def acuracia_torch(modelo, X, y):
    modelo.eval()  # modo avaliação (desliga dropout etc.; aqui não temos, mas é boa prática)
    with torch.no_grad():  # não calcula gradientes (mais rápido, economiza memória)
        logits = modelo(X)
        preds = torch.argmax(logits, dim=1)
        acc = (preds == y).float().mean().item()
    modelo.train()  # volta pro modo treino
    return acc

# ---- Histórico ----
hist_torch = {"loss": [], "acc_tr": [], "acc_val": []}

n = X_train_dev.shape[0]
print(f"Treinando MLP PyTorch | {EPOCAS} épocas | batch {BATCH_SIZE} | device {device}\n")

for ep in range(EPOCAS):
    # Embaralhar a ordem a cada época
    ordem = torch.randperm(n, device=device)

    perda_acum = 0.0
    n_batches = 0

    for ini in range(0, n, BATCH_SIZE):
        idx = ordem[ini:ini+BATCH_SIZE]
        Xb = X_train_dev[idx]
        yb = y_train_dev[idx]

        # --- O ciclo de treino do PyTorch (compare com a Etapa 1!) ---
        otimizador.zero_grad()      # zera gradientes antigos
        logits = modelo(Xb)         # forward
        perda = criterio(logits, yb)  # calcula a perda
        perda.backward()            # backward AUTOMÁTICO (autograd!)
        otimizador.step()           # atualiza os pesos

        perda_acum += perda.item()
        n_batches += 1

    # Métricas da época
    loss_ep = perda_acum / n_batches
    acc_tr  = acuracia_torch(modelo, X_train_dev, y_train_dev)
    acc_val = acuracia_torch(modelo, X_val_dev, y_val_dev)

    hist_torch["loss"].append(loss_ep)
    hist_torch["acc_tr"].append(acc_tr)
    hist_torch["acc_val"].append(acc_val)

    print(f"Época {ep+1:2d}/{EPOCAS} | perda {loss_ep:.4f} | "
          f"acc treino {acc_tr:.4f} | acc val {acc_val:.4f}")

print("\nTreino concluído.")

Treinando MLP PyTorch | 20 épocas | batch 128 | device cuda

Época  1/20 | perda 1.7627 | acc treino 0.4052 | acc val 0.3975
Época  2/20 | perda 1.5107 | acc treino 0.4394 | acc val 0.4353
Época  3/20 | perda 1.4006 | acc treino 0.5006 | acc val 0.4930
Época  4/20 | perda 1.3338 | acc treino 0.4166 | acc val 0.4079
Época  5/20 | perda 1.3091 | acc treino 0.3427 | acc val 0.3370
Época  6/20 | perda 1.2833 | acc treino 0.5262 | acc val 0.5185
Época  7/20 | perda 1.2716 | acc treino 0.4460 | acc val 0.4368
Época  8/20 | perda 1.2785 | acc treino 0.5333 | acc val 0.5298
Época  9/20 | perda 1.2574 | acc treino 0.5115 | acc val 0.5146
Época 10/20 | perda 1.2294 | acc treino 0.5483 | acc val 0.5401
Época 11/20 | perda 1.2073 | acc treino 0.5178 | acc val 0.5111
Época 12/20 | perda 1.2056 | acc treino 0.5627 | acc val 0.5550
Época 13/20 | perda 1.1756 | acc treino 0.5581 | acc val 0.5501
Época 14/20 | perda 1.1866 | acc treino 0.5506 | acc val 0.5445
Época 15/20 | perda 1.1627 | acc treino 0.5

In [5]:
# ============================================================
# ETAPA 2 — BLOCO 4B: Treino para prova de equivalência (mesmos 20k da Etapa 1)
# ============================================================
import torch.optim as optim

# ---- Mesmos hiperparâmetros da Etapa 1 ----
EPOCAS     = 20
BATCH_SIZE = 128
LR         = 0.01
BETA       = 0.9
TAM_SUBSET = 20000   # << igual à Etapa 1

# ---- Recriar o modelo do zero (pesos novos) com a mesma seed ----
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

modelo_eq = MLP_PyTorch([2352, 256, 128, 9]).to(device)

criterio   = nn.CrossEntropyLoss()
otimizador = optim.SGD(modelo_eq.parameters(), lr=LR, momentum=BETA)

# ---- Usar o MESMO subconjunto de 20k da Etapa 1 ----
X_sub = X_train_t[:TAM_SUBSET].to(device)
y_sub = y_train_t[:TAM_SUBSET].to(device)
X_val_dev = X_val_t.to(device)
y_val_dev = y_val_t.to(device)

def acuracia_torch(modelo, X, y):
    modelo.eval()
    with torch.no_grad():
        preds = torch.argmax(modelo(X), dim=1)
        acc = (preds == y).float().mean().item()
    modelo.train()
    return acc

hist_eq = {"loss": [], "acc_tr": [], "acc_val": []}
n = X_sub.shape[0]
print(f"Treino de equivalência | {n} imagens | {EPOCAS} épocas | device {device}\n")

for ep in range(EPOCAS):
    ordem = torch.randperm(n, device=device)
    perda_acum, n_batches = 0.0, 0

    for ini in range(0, n, BATCH_SIZE):
        idx = ordem[ini:ini+BATCH_SIZE]
        Xb, yb = X_sub[idx], y_sub[idx]

        otimizador.zero_grad()
        logits = modelo_eq(Xb)
        perda = criterio(logits, yb)
        perda.backward()
        otimizador.step()

        perda_acum += perda.item()
        n_batches += 1

    loss_ep = perda_acum / n_batches
    acc_tr  = acuracia_torch(modelo_eq, X_sub, y_sub)
    acc_val = acuracia_torch(modelo_eq, X_val_dev, y_val_dev)

    hist_eq["loss"].append(loss_ep)
    hist_eq["acc_tr"].append(acc_tr)
    hist_eq["acc_val"].append(acc_val)

    print(f"Época {ep+1:2d}/{EPOCAS} | perda {loss_ep:.4f} | "
          f"acc treino {acc_tr:.4f} | acc val {acc_val:.4f}")

# ---- Comparação final com a Etapa 1 ----
acc_val_pytorch = hist_eq["acc_val"][-1]
acc_val_numpy   = 0.5012   # << resultado da última época da sua Etapa 1

print("\n" + "="*50)
print("PROVA DE EQUIVALÊNCIA (mesmas condições)")
print("="*50)
print(f"Acurácia val NumPy  (Etapa 1): {acc_val_numpy:.4f}")
print(f"Acurácia val PyTorch (Etapa 2): {acc_val_pytorch:.4f}")
diff = abs(acc_val_pytorch - acc_val_numpy) * 100
print(f"Diferença: {diff:.2f} p.p.")
print("Critério (<= 2 p.p.):", "ATENDIDO ✓" if diff <= 2 else "verificar (pode variar por seed)")

Treino de equivalência | 20000 imagens | 20 épocas | device cuda

Época  1/20 | perda 2.0807 | acc treino 0.2427 | acc val 0.2408
Época  2/20 | perda 1.7767 | acc treino 0.2993 | acc val 0.2961
Época  3/20 | perda 1.6802 | acc treino 0.3946 | acc val 0.3858
Época  4/20 | perda 1.6305 | acc treino 0.4025 | acc val 0.3976
Época  5/20 | perda 1.5959 | acc treino 0.2869 | acc val 0.2835
Época  6/20 | perda 1.5806 | acc treino 0.4303 | acc val 0.4206
Época  7/20 | perda 1.5159 | acc treino 0.4479 | acc val 0.4421
Época  8/20 | perda 1.4763 | acc treino 0.4319 | acc val 0.4215
Época  9/20 | perda 1.4561 | acc treino 0.4587 | acc val 0.4509
Época 10/20 | perda 1.4130 | acc treino 0.4557 | acc val 0.4476
Época 11/20 | perda 1.3941 | acc treino 0.4652 | acc val 0.4614
Época 12/20 | perda 1.3975 | acc treino 0.5239 | acc val 0.5119
Época 13/20 | perda 1.3540 | acc treino 0.4869 | acc val 0.4738
Época 14/20 | perda 1.3922 | acc treino 0.4885 | acc val 0.4744
Época 15/20 | perda 1.3384 | acc trein

In [6]:
# ============================================================
# ETAPA 2 — BLOCO 4C: Equivalência robusta (média das últimas épocas)
# ============================================================
import numpy as np

# Médias das últimas 5 épocas de cada implementação
acc_pytorch_media = np.mean(hist_eq["acc_val"][-5:])

# Cole aqui as 5 últimas acc_val da sua Etapa 1 (NumPy).
# Do seu output da Etapa 1, as últimas 5 foram:
# ép16=0.5007, ép17=0.5139, ép18=0.5012, ép19=0.4761, ép20=0.5012
acc_numpy_ultimas = [0.5007, 0.5139, 0.5012, 0.4761, 0.5012]
acc_numpy_media = np.mean(acc_numpy_ultimas)

print("="*55)
print("PROVA DE EQUIVALÊNCIA — média das últimas 5 épocas")
print("="*55)
print(f"Acc val média NumPy  (Etapa 1): {acc_numpy_media:.4f}")
print(f"Acc val média PyTorch (Etapa 2): {acc_pytorch_media:.4f}")
diff = abs(acc_pytorch_media - acc_numpy_media) * 100
print(f"Diferença média: {diff:.2f} p.p.")
print("Critério (<= 2 p.p.):", "ATENDIDO ✓" if diff <= 2 else "ainda acima — discutir")

PROVA DE EQUIVALÊNCIA — média das últimas 5 épocas
Acc val média NumPy  (Etapa 1): 0.4986
Acc val média PyTorch (Etapa 2): 0.4543
Diferença média: 4.43 p.p.
Critério (<= 2 p.p.): ainda acima — discutir


In [7]:
# ============================================================
# ETAPA 2 — BLOCO 4D: Equivalência final (LR=0.005, mesmos 20k)
# ============================================================
import torch.optim as optim
import numpy as np

EPOCAS, BATCH_SIZE, LR, BETA, TAM_SUBSET = 20, 128, 0.005, 0.9, 20000

torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

modelo_eq = MLP_PyTorch([2352, 256, 128, 9]).to(device)
criterio   = nn.CrossEntropyLoss()
otimizador = optim.SGD(modelo_eq.parameters(), lr=LR, momentum=BETA)

X_sub = X_train_t[:TAM_SUBSET].to(device)
y_sub = y_train_t[:TAM_SUBSET].to(device)
X_val_dev = X_val_t.to(device)
y_val_dev = y_val_t.to(device)

def acuracia_torch(modelo, X, y):
    modelo.eval()
    with torch.no_grad():
        preds = torch.argmax(modelo(X), dim=1)
        acc = (preds == y).float().mean().item()
    modelo.train()
    return acc

hist_eq = {"loss": [], "acc_tr": [], "acc_val": []}
n = X_sub.shape[0]
print(f"Treino equivalência | {n} imagens | LR={LR} | {EPOCAS} épocas\n")

for ep in range(EPOCAS):
    ordem = torch.randperm(n, device=device)
    perda_acum, n_batches = 0.0, 0
    for ini in range(0, n, BATCH_SIZE):
        idx = ordem[ini:ini+BATCH_SIZE]
        Xb, yb = X_sub[idx], y_sub[idx]
        otimizador.zero_grad()
        perda = criterio(modelo_eq(Xb), yb)
        perda.backward()
        otimizador.step()
        perda_acum += perda.item(); n_batches += 1
    loss_ep = perda_acum / n_batches
    acc_tr  = acuracia_torch(modelo_eq, X_sub, y_sub)
    acc_val = acuracia_torch(modelo_eq, X_val_dev, y_val_dev)
    hist_eq["loss"].append(loss_ep)
    hist_eq["acc_tr"].append(acc_tr)
    hist_eq["acc_val"].append(acc_val)
    print(f"Época {ep+1:2d}/{EPOCAS} | perda {loss_ep:.4f} | acc tr {acc_tr:.4f} | acc val {acc_val:.4f}")

# ---- Comparação: ponto final E média das últimas 5 ----
# Cole as acc_val das últimas 5 épocas da sua Etapa 1 com LR=0.005:
numpy_ult5 = [0.5000, 0.5183, 0.4969, 0.4978, 0.4243]   # épocas 16-20
numpy_final = 0.4243   # época 20

pytorch_ult5 = hist_eq["acc_val"][-5:]
pytorch_final = hist_eq["acc_val"][-1]

print("\n" + "="*55)
print("PROVA DE EQUIVALÊNCIA (LR=0.005)")
print("="*55)
diff_final = abs(pytorch_final - numpy_final) * 100
diff_media = abs(np.mean(pytorch_ult5) - np.mean(numpy_ult5)) * 100
print(f"[Ponto final]  NumPy {numpy_final:.4f} | PyTorch {pytorch_final:.4f} | dif {diff_final:.2f} p.p.")
print(f"[Média últ. 5] NumPy {np.mean(numpy_ult5):.4f} | PyTorch {np.mean(pytorch_ult5):.4f} | dif {diff_media:.2f} p.p.")
print("\nCritério (<= 2 p.p.):")
print("  Por ponto final:", "ATENDIDO ✓" if diff_final <= 2 else "acima")
print("  Por média 5 ép.:", "ATENDIDO ✓" if diff_media <= 2 else "acima")

Treino equivalência | 20000 imagens | LR=0.005 | 20 épocas

Época  1/20 | perda 2.1500 | acc tr 0.2347 | acc val 0.2357
Época  2/20 | perda 1.9130 | acc tr 0.2629 | acc val 0.2616
Época  3/20 | perda 1.7591 | acc tr 0.3721 | acc val 0.3680
Época  4/20 | perda 1.6793 | acc tr 0.3627 | acc val 0.3600
Época  5/20 | perda 1.6522 | acc tr 0.3920 | acc val 0.3845
Época  6/20 | perda 1.6235 | acc tr 0.4049 | acc val 0.3964
Época  7/20 | perda 1.5902 | acc tr 0.4248 | acc val 0.4169
Época  8/20 | perda 1.5711 | acc tr 0.4267 | acc val 0.4138
Época  9/20 | perda 1.5346 | acc tr 0.4133 | acc val 0.4055
Época 10/20 | perda 1.5216 | acc tr 0.4552 | acc val 0.4451
Época 11/20 | perda 1.4918 | acc tr 0.4366 | acc val 0.4325
Época 12/20 | perda 1.4676 | acc tr 0.4765 | acc val 0.4582
Época 13/20 | perda 1.4417 | acc tr 0.4819 | acc val 0.4690
Época 14/20 | perda 1.4466 | acc tr 0.4532 | acc val 0.4473
Época 15/20 | perda 1.4101 | acc tr 0.4647 | acc val 0.4496
Época 16/20 | perda 1.3964 | acc tr 0.47